# Convergence per Attack per Algorithm

In [1]:
# Set environment variables necessary for running parallel processes
import os
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning"

In [2]:
%load_ext autoreload
import sys, warnings, time, json, copy
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pandas as pd
import seaborn as sns
from pathlib import Path
from fastnanoid import generate
from datetime import datetime
from sklearn.utils.parallel import Parallel, delayed

In [3]:
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
lib_path = project_root / "lib"
sys.path.insert(0, str(lib_path))        
sys.path.insert(0, str(project_root))   

In [4]:
%autoreload 2
from lib.graph_factory import GraphFactory, add_graph_plot
from lib.proj_const_estimator import ProjConstEstimator
from lib.utils import fetch_dataset, get_alphas, rng, seed, save_to_pkl, load_from_pkl
from lib.classifier import ByzClassifier, load_run, save_run
from lib.system import SystemSimulator
from lib.metrics import MetricsCalculator
from lib.config import BASE_CONF, NUM_NODES

**Helper Functions**

In [5]:
def convert(o):
    if isinstance(o, np.generic):
        return o.item()
    raise TypeError(f'{type(o)} not serializable')

In [6]:
DATA_CACHE = os.path.join(Path().resolve(), 'mnist_pca.npz') # Cached so threads can access it w/o race cond.
def load_dataset(path=DATA_CACHE):
    if os.path.exists(path):
        z = np.load(path)
        return {'num_classes': int(z['num_classes']),
                'X_train': z['X_train'], 'y_train': z['y_train'],
                'X_test':  z['X_test'],  'y_test':  z['y_test']}
    gd = fetch_dataset('MNIST')
    tmp = path + '.tmp.npz'
    np.savez(tmp, num_classes=gd['num_classes'],
             X_train=gd['X_train'], y_train=gd['y_train'],
             X_test=gd['X_test'],   y_test=gd['y_test'])
    os.replace(tmp, path)
    return gd

In [7]:
def _paths():
    import sys
    for p in (str(project_root), str(lib_path)):
        if p not in sys.path: sys.path.insert(0, p)

**Processes**

In [8]:
def clf_stage(g, config_g, run_dir, b):
    _paths()
    import os, copy
    from lib.graph_factory import GraphFactory
    from lib.proj_const_estimator import ProjConstEstimator
    from lib.classifier import ByzClassifier, load_run, save_run
    from lib.utils import fetch_dataset, seed

    gd = load_dataset() 
    gf = GraphFactory(config_g['train']['num_nodes'], b)
    clf = ByzClassifier(config_g, gd, gf)

    pkl_file = os.path.join(run_dir, f"clf_sim_data_{g}_{b}.pkl.gz")
    if os.path.exists(pkl_file):
        proj_const = load_run(clf, pkl_file)['proj_const']
    else:
        config_g['graph_args']['MH_target_pi'] = (1 - config['train']['gamma_C']) * np.full(NUM_NODES, 1.0/NUM_NODES)
        
        pce = ProjConstEstimator(config_g, gd, gf)
        pce.configure(config_g, seed('proj-const-estimator'))
        proj_const = pce.estimate()
        
        clf.run_simulations(config_g, proj_const, seed('byz-clf','sim'), is_printing=False)
        save_run(clf, pkl_file, {'proj_const': proj_const})

    clf_est, clf_preproc = clf.fit()
    clf_op_pt = clf.calc_optimal_op_pt()
    gamma_sys = np.clip(clf_op_pt['C_fpr'] + (np.linspace(-1, 1, NUM_NODES) * config['sys']['fpr_spread']), 0, 0.95)
    clf_thr = clf.threshold_at_fpr(gamma_sys)
    
    return g, dict(proj_const=proj_const, est=clf_est, preproc=clf_preproc,
                   op_pt=clf_op_pt, clf_thr=clf_thr, metrics=clf.test(), gamma_sys=gamma_sys)

In [9]:
def sim_task(g, config_g, atk, s, proj_const, preproc, est, gamma_sys, params_C, algs, b): 
    _paths()
    import copy
    from lib.graph_factory import GraphFactory
    from lib.system import SystemSimulator
    from lib.metrics import MetricsCalculator
    from lib.utils import fetch_dataset, rng, seed

    gd = load_dataset() 
    gf = GraphFactory(config_g['train']['num_nodes'], b)
    ss = SystemSimulator(config_g, gd, gf)

    pi_t = (1-gamma_sys) * (np.full(NUM_NODES, 1.0/NUM_NODES))
    config_g['graph_args']['MH_target_pi'] = pi_t
    ss.init_simulation(config_g, proj_const, preproc, est, params_C, seed('sys', s), is_printing_logs=False)
    df = ss.simulate(dict(algorithms=algs, atk_type=atk, threat_model='T3', seed=s))
    
    consts = None
    if s == SEEDS[0]:
        mc = MetricsCalculator(config_g, gd, rng('metrics-calculator'))
        consts = mc(ss.get_sim_config(), ss.models, params_C['C_fpr'], params_C['C_fnr'], proj_const, pi_t)
        for k in ('pi', 'x_opt', 'x_pi_opt'): consts.pop(k, None)
    return g, atk, s, df, consts

**Experiment Configuration**

In [10]:
%%capture
RUN_DIR = os.path.join(Path().resolve(), "baseline")
IMAGES_DIR = os.path.join(RUN_DIR, "images")
config = copy.deepcopy(BASE_CONF)

# Communication Graph
b = 5
config['graph_weights'] = 'MH_gen'
config['graph_args'] = {
    'ba_m': 5,
    'ws_k': 6,
    'ws_p': 0.1,
    'rand_reg_deg':10,
    'geom_radius':0.4,
    'er_p':min(1.0, 3.0 * np.log(NUM_NODES) / NUM_NODES),
}

# Classifier Training 
config['train']['b'] = b
config['train']['fpr_mean'] = 0.2
config['train']['fpr_spread'] = 0.05
config['train']['gamma_C'] = np.clip((config['train']['fpr_spread']*np.linspace(-1,1,NUM_NODES)) + \
                                     config['train']['fpr_mean'], 0, 0.95)

# System Simulation 
config['sys']['b'] = b
config['sys']['fpr_spread'] = 0.003
config['sys']['dropout_ub'] = {
    'erdos-renyi': 0.1232,
    'random-regular': 0.0108,
    'watts-strogatz': 0.077,
    'geometric': 0.0789,
    'barabasi-albert': 0.0535
}

# Miscellaneous
config['train']['clf_model'] = 'xgb'
config['data_heterogeneity'] = 100
config['reg_param'] = 1

# Notebook Variables
graphs = ['random-regular','watts-strogatz', 'geometric', 'erdos-renyi', ]
ATKS = ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM']
ALGS = ['RDSGD', 'ORACLE', 'IOS', 'SCC', 'TriMean', 'CooMed']
ALGS_TEST = ['RDSGD', 'ORACLE']
COLORS = dict(zip(ALGS, ['C0', 'C1', 'C2', 'C3', 'C4', 'C5']))
SEEDS = [111,333,777]
DEBUG = True
plots = [plt.subplots(1,len(ATKS),figsize=(15,3), sharey=True) for g in graphs]
ta_tables = dict()

In [11]:
# Save experiment configuration
payload = copy.deepcopy(config)
del payload['train']['gamma_C']
with open(os.path.join(RUN_DIR, f'config.json'), 'w') as f:
    json.dump(payload, f, indent=2, default=convert)

In [12]:
CACHE = {}
for g in graphs:
    cfg = copy.deepcopy(config)
    cfg['graph_type'] = g
    cfg['sys']['dropout'] = min(config['sys']['dropout_ub'][g], 0.2)
    os.makedirs(os.path.join(RUN_DIR, g), exist_ok=True)
    CACHE[g] = cfg

In [13]:
# Train Byzantine classifier in parallel over 5 topologies
res1 = Parallel(n_jobs=5, backend='loky', inner_max_num_threads=4, verbose=10)(
    delayed(clf_stage)(g, CACHE[g], RUN_DIR, b) for g in graphs)
P1 = dict(res1)

[Parallel(n_jobs=5)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=5)]: Done   1 tasks      | elapsed:   35.3s
[Parallel(n_jobs=5)]: Done   2 out of   4 | elapsed:   35.4s remaining:   35.4s
[Parallel(n_jobs=5)]: Done   4 out of   4 | elapsed:   38.8s finished


In [14]:
# Run simulations in parallel over 5 topologies
jobs = [(g, atk, s) for g in graphs for atk in ATKS for s in SEEDS]
res2 = Parallel(n_jobs=15, backend='loky', inner_max_num_threads=1, verbose=10)(
    delayed(sim_task)(g, CACHE[g], atk, s, P1[g]['proj_const'],
                      P1[g]['preproc'], P1[g]['est'], P1[g]['gamma_sys'],
                      dict(C_fpr=P1[g]['clf_thr']['fpr_ach'],
                           C_fnr=P1[g]['clf_thr']['fnr_ach'],
                           C_tau=P1[g]['clf_thr']['C_tau']),
                      ALGS, b)
    for g, atk, s in jobs)

[Parallel(n_jobs=15)]: Using backend LokyBackend with 15 concurrent workers.
/home/linux/env/ml/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/linux/env/ml/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/linux/env/ml/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/li

In [15]:
# Aggregate simulation results and save to disk
R = {(g, atk, s): (df, consts) for g, atk, s, df, consts in res2}
rows, met_rows, clf_rows = [], [], []

for g_idx, g in enumerate(graphs):
    fig, ax = plots[g_idx]
    clf_rows.append({'graph': g, 'proj_const': P1[g]['proj_const'],
                     **P1[g]['op_pt'], **P1[g]['metrics']})
    for atk_idx, atk in enumerate(ATKS):
        for s in SEEDS:
            df, consts = R[(g, atk, s)]
            first = (s == SEEDS[0])
            if first:
                df.to_csv(os.path.join(RUN_DIR, g, f'sys_sim_df_{g}_{b}_{atk}.csv'))
                met_rows.append({**consts, 'atk': atk, 'graph': g})
            for alg in ALGS:
                col = 'test_acc_pi' if alg == 'RDSGD' else 'test_acc'
                rows.append({'alg': alg, 'atk': atk, 'seed': s,
                             'graph': g, 'test_acc': df.loc[alg, col].iloc[-1]})
                if first:
                    V = df.loc[alg, 'C_unif'] + np.pow(df.loc[alg, 'opt_gap'], 2)
                    ax[atk_idx].plot(V, color=COLORS[alg])
                    ax[atk_idx].set(title=f'({g}, {atk})', yscale='log', xlabel='Iteration (k)')
                    if(atk_idx == 0): ax[atk_idx].set_ylabel("Lyapunov Function $V^{k}$")
                    else: ax[atk_idx].set_ylabel("")
                    
    os.makedirs(IMAGES_DIR, exist_ok=True)
    fig.savefig(os.path.join(IMAGES_DIR, f'conv_plots_{g}_{b}.png'))

pd.DataFrame(clf_rows).set_index('graph').to_csv(os.path.join(RUN_DIR, f'clf_metrics_{b}.csv'))
pd.DataFrame(met_rows).set_index(['graph','atk']).to_csv(os.path.join(RUN_DIR, f'sys_metrics_{b}.csv'))

df_all = pd.DataFrame(rows)
for g in graphs:
    ta_tables[g] = (df_all[df_all.graph == g]
                    .groupby(['atk','alg'])['test_acc'].agg(['mean','std'])
                    .unstack('alg'))
    ta_tables[g].to_csv(os.path.join(RUN_DIR, g, "test_accuracy_summary.csv"))
    print(f"{g} graph test accuracy table"); display(ta_tables[g])

random-regular graph test accuracy table


mean                                                    \
alg           CooMed       IOS    ORACLE     RDSGD       SCC   TriMean   
atk                                                                      
ALIE        0.851025  0.850087  0.851847  0.851587  0.851342  0.851400   
IPM         0.851241  0.849942  0.851934  0.851587  0.851818  0.851515   
gaussian    0.839740  0.458730  0.851890  0.851645  0.830996  0.730231   
label_flip  0.848543  0.850303  0.851602  0.851558  0.844141  0.849495   
sign_flip   0.844444  0.843290  0.851573  0.847677  0.844416  0.846984   

                 std                                                    
alg           CooMed       IOS    ORACLE     RDSGD       SCC   TriMean  
atk                                                                     
ALIE        0.001147  0.000976  0.000282  0.001139  0.001356  0.001325  
IPM         0.000368  0.000650  0.000328  0.000951  0.000909  0.000680  
gaussian    0.013540  0.255298  0.000402  0.001053  0.002731  0.167332  
label_flip  0.000943  0.000385  0.000417  0.001089  0.002381  0.000929  
sign_flip   0.000451  0.000715  0.000933  0.000631  0.000845  0.000548

watts-strogatz graph test accuracy table


mean                                                    \
alg           CooMed       IOS    ORACLE     RDSGD       SCC   TriMean   
atk                                                                      
ALIE        0.850361  0.849798  0.852049  0.851443  0.851645  0.851241   
IPM         0.851457  0.850967  0.851977  0.851645  0.852136  0.851602   
gaussian    0.536753  0.381962  0.852078  0.851789  0.831587  0.561991   
label_flip  0.848961  0.850029  0.851111  0.851241  0.844113  0.850317   
sign_flip   0.844127  0.848254  0.851241  0.845830  0.843968  0.847431   

                 std                                                    
alg           CooMed       IOS    ORACLE     RDSGD       SCC   TriMean  
atk                                                                     
ALIE        0.001117  0.002076  0.000846  0.001346  0.001387  0.000746  
IPM         0.000714  0.000282  0.000565  0.001391  0.000934  0.000606  
gaussian    0.266566  0.171460  0.000709  0.001039  0.002479  0.217105  
label_flip  0.000888  0.000619  0.000347  0.001054  0.002219  0.000813  
sign_flip   0.000363  0.002667  0.000634  0.001359  0.000846  0.000572

geometric graph test accuracy table


mean                                                    \
alg           CooMed       IOS    ORACLE     RDSGD       SCC   TriMean   
atk                                                                      
ALIE        0.850952  0.851313  0.852439  0.851400  0.851789  0.851227   
IPM         0.851126  0.850678  0.852468  0.851342  0.852035  0.851183   
gaussian    0.654257  0.358759  0.852049  0.851905  0.831645  0.433362   
label_flip  0.848600  0.850678  0.851097  0.851227  0.844055  0.849322   
sign_flip   0.842208  0.844315  0.850851  0.845180  0.843795  0.844502   

                 std                                                    
alg           CooMed       IOS    ORACLE     RDSGD       SCC   TriMean  
atk                                                                     
ALIE        0.001710  0.002227  0.000982  0.000815  0.001277  0.001775  
IPM         0.001275  0.002060  0.000949  0.000802  0.000740  0.001064  
gaussian    0.185272  0.167536  0.001237  0.001072  0.003415  0.290068  
label_flip  0.000780  0.002037  0.001273  0.001342  0.002187  0.000815  
sign_flip   0.002155  0.004405  0.000477  0.000307  0.000694  0.001093

erdos-renyi graph test accuracy table


mean                                                    \
alg           CooMed       IOS    ORACLE     RDSGD       SCC   TriMean   
atk                                                                      
ALIE        0.850808  0.850072  0.852626  0.851688  0.851717  0.851212   
IPM         0.851775  0.850447  0.852583  0.851905  0.851991  0.852020   
gaussian    0.850231  0.311890  0.852367  0.851775  0.831847  0.722771   
label_flip  0.848557  0.850289  0.851688  0.850952  0.844242  0.849437   
sign_flip   0.843434  0.841255  0.851703  0.847359  0.844156  0.846032   

                 std                                                    
alg           CooMed       IOS    ORACLE     RDSGD       SCC   TriMean  
atk                                                                     
ALIE        0.000325  0.000765  0.000804  0.001128  0.001225  0.000900  
IPM         0.001024  0.002168  0.000824  0.000901  0.000795  0.001327  
gaussian    0.001143  0.009437  0.000704  0.001281  0.002907  0.060745  
label_flip  0.000631  0.002316  0.000229  0.000996  0.002367  0.000998  
sign_flip   0.000846  0.004119  0.000832  0.001164  0.000779  0.000929